<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/ADPs/Practica/%D0%98%D1%82%D0%BE%D0%B3%D0%BE%D0%B2%D1%8B%D0%B9_%D0%BF%D1%80%D0%BE%D0%B5%D0%BA%D1%82.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ТЕХНИЧЕСКОЕ ЗАДАНИЕ
## Итоговый проект: «Мультиагентный ИИ-ассистент психолога»

---

## 1. Общие сведения

| Параметр | Значение |
|---|---|
| **Название проекта** | Мультиагентный ИИ-ассистент по выбранной психологической теме |
| **Тип работы** | Индивидуальный или парный проект (пара — 2 человека) |
| **Срок выполнения** | 8 недель |
| **Итоговый продукт** | Работающее веб-приложение на Streamlit + публичная ссылка на Hugging Face Spaces + репозиторий на GitHub |
| **База** | Материалы работ 1–3 (чат с Ollama, RAG, tool calling, мультиагентная цепочка) |
| **Язык программирования** | Python 3.10+ |
| **Среда разработки** | VS Code, терминал PowerShell |

**Цель проекта:** собрать легальный корпус психологических данных по выбранной теме, построить на его основе мультиагентную систему (Ответчик → Критик → Корректор → Судья), обернуть её в веб-интерфейс с дисклеймером, зафиксировать все этапы в Git (коммиты от каждого участника) и опубликовать демо-версию в облаке.

**Задачи:**
1. Собрать и структурировать легальные данные по теме.
2. При необходимости — написать вспомогательный парсер PDF → текст/JSONL.
3. Реализовать инструменты поиска по данным.
4. Реализовать четырёх агентов с разными ролями.
5. Создать Streamlit-интерфейс с дисклеймером и вкладкой оценки.
6. Зафиксировать этапы в Git (коммиты от каждого участника пары).
7. Опубликовать демо-версию на Hugging Face Spaces.
8. Подготовить документацию, отчёт и защиту.

---

## 2. Требования к данным

### 2.1. Источник данных

Студент выбирает **одну тему** из списка направлений психологии (когнитивная, клиническая, личность, социальная, развития, труда, здоровья, психотерапия, нейро, позитивная) и **один источник**:

- книга или глава из книги по психологии (PDF или текст) — **public domain, open access или материал, на который у вас есть право**;
- набор из 5–10 научных статей (абстракты или полные тексты, желательно open access);
- анонимизированные ответы на опросы (с согласия респондентов).

**Требования к источнику:**
- **Уникальность:** тема уникальна в группе; источник уникален, даже если тема одна.
- **Легальность:** пиратские PDF книг и закрытых статей **запрещено** выкладывать в публичный репозиторий и на HF Space. Если используете библиотечный доступ — в репозиторий кладите только ссылки и/или абстракты, а `source.txt` формируйте из открытого текста.
- **Извлекаемость:** если используется PDF, текст должен извлекаться через `pypdf`. Сканированные PDF без OCR не принимаются.
- **Объём:** не менее 3 страниц (или 5000 символов для `.txt`).

**Роль `source.pdf` / `source.txt` в проекте.** Это **первоисточник**, из которого студент **вручную** формирует `glossary.json` и `notes/*.txt`. Код **не читает** `source.txt` напрямую — файл нужен как доказательство, что определения и заметки не выдуманы, а взяты из материала. На защите студент показывает, откуда взято определение термина.

### 2.2. Структура папки `data/`

```
data/
├── source.pdf (или source.txt)    ← первоисточник (не читается кодом)
├── glossary.json                   ← глоссарий терминов (10+)
├── notes/                          ← 3–5 тематических заметок
│   ├── definition.txt
│   ├── examples.txt
│   ├── research.txt
│   └── criticism.txt
└── chunks.jsonl                    ← (опционально) вспомогательный формат
```

**Пояснение по `chunks.jsonl`:** это **вспомогательный** формат (по одной JSON-записи на строку: `{"id": ..., "text": ..., "source": ...}`). Он **не заменяет** обязательные `glossary.json` и `notes/*.txt`, используется только для внутренних экспериментов. В чек-лист сдачи не входит.

### 2.3. Требования к `glossary.json`

- Формат: JSON-объект, ключ — термин (строка), значение — определение (строка).
- **Минимум 10 терминов** по теме.
- Кодировка UTF-8.
- Определения — краткие (1–3 предложения), научные, без воды, **сверенные с источником**.
- Запрещено генерировать глоссарий полностью «из головы» без сверки с материалом.

**Пример:**

```json
{
  "катастрофизация": "Когнитивное искажение: представление о том, что произойдёт самое худшее из возможного, без учёта других исходов.",
  "сверхобобщение": "Когнитивное искажение: вывод о том, что одно негативное событие будет повторяться всегда.",
  "персонализация": "Когнитивное искажение: принятие на себя ответственности за события, которые от вас не зависят."
}
```

### 2.4. Требования к `notes/*.txt`

- **3–5 файлов** по подтемам (например, `definition.txt`, `examples.txt`, `research.txt`, `criticism.txt`).
- Каждый файл — 1–3 абзаца **связного текста** по подтеме (не сырые чанки из парсера).
- Кодировка UTF-8.
- Имя файла — латиницей, без пробелов, с расширением `.txt`.

### 2.5. Вспомогательный парсер (опционально, но приветствуется)

Если вы скачиваете PDF-статьи/главы, можно написать скрипт `parse_source.py`:

- вход: `data/source.pdf` (или несколько PDF в `data/raw/`);
- выход: `data/source.txt` (очищенный текст) и опционально `data/chunks.jsonl`;
- использует `pypdf`;
- **проверяет, что извлечено не менее 5000 символов**;
- если извлечено меньше 5000 символов — **выводит предупреждение и не сохраняет `source.txt`** (чтобы студент не индексировал пустоту);
- **не обязателен** в чек-листе, но упрощает работу и показывает навык.

`pypdf` в этом случае добавляется в `requirements.txt`.

---

## 3. Требования к коду

### 3.1. Общая структура проекта

```
<корень проекта>/
├── data/
│   ├── source.pdf (или .txt)
│   ├── glossary.json
│   ├── notes/
│   └── chunks.jsonl                ← опционально
├── parse_source.py                 ← опционально (парсер PDF→txt/jsonl)
├── tools.py
├── test_tools.py
├── agents.py
├── multi_agent_core.py
├── app.py
├── test_questions.json
├── requirements.txt
├── README.md
└── REPORT.md
```

### 3.2. Файл `tools.py` — инструменты

**Назначение:** функции, которые вызывают агенты для поиска в данных.

**Обязательные функции:**

| Функция | Вход | Выход | Поведение |
|---|---|---|---|
| `lookup_glossary(term: str) -> str` | термин | определение или сообщение «не найден» | Ищет в `data/glossary.json`. Если термина нет — возвращает `«Термин <term> не найден в глоссарии.»` |
| `read_notes(topic: str) -> str` | тема | текст заметки или сообщение «не найдено» | Читает `data/notes/<topic>.txt`. Если файла нет — `«Заметок по теме <topic> не найдено.»` |

**Опционально (по желанию):** `add_numbers(a: float, b: float) -> float` — простая арифметика для проверки tool calling. Если реализуете — добавьте его в `TOOLS` и в тестовые вопросы.

**Обязательная константа `TOOLS`:** список описаний функций в формате JSON Schema (OpenAI-совместимый):

```python
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "lookup_glossary",
            "description": "Ищет определение психологического термина в глоссарии.",
            "parameters": {
                "type": "object",
                "required": ["term"],
                "properties": {
                    "term": {"type": "string", "description": "Термин, например «катастрофизация»"}
                }
            }
        }
    },
    # ... описание read_notes (и add_numbers, если реализован)
]
```

**Требования к коду:**
- комментарии на русском;
- без классов и декораторов;
- обработка ошибок: файл не найден, JSON невалиден, пустой ввод;
- **никаких `print()`**.

### 3.3. Файл `test_tools.py` — тесты инструментов

**Обязательный минимум — 5 тестов:**

1. `lookup_glossary` с существующим термином → возвращает определение.
2. `lookup_glossary` с несуществующим термином → «не найден».
3. `lookup_glossary` с пустой строкой → «не найден».
4. `read_notes` с существующим файлом → возвращает текст.
5. `read_notes` с несуществующим файлом → «не найдено».

**Если реализован `add_numbers`** — добавьте **6-й тест**: `add_numbers(2, 3)` возвращает `5`. Если `add_numbers` не реализован — 6-й тест не нужен.

**Формат:** функции `test_*()` с `assert`. Запуск — `python test_tools.py` (без `pytest`).

### 3.4. Файл `agents.py` — логика агентов

**Назначение:** четыре функции-агента, каждая — отдельный вызов модели с разным системным промптом.

#### 3.4.1. `answer_agent(question: str, model: str) -> dict`

**Роль:** Ответчик. Отвечает на вопрос, вызывая инструменты при необходимости.

**Возвращает:**

```python
{
    "answer": str,
    "steps": [
        {
            "step": 1,
            "tool_calls": [{"name": ..., "args": ..., "result": ...}],
            "assistant_content": str
        },
        ...
    ]
}
```

**Требования:**
- цикл по `tool_calls` (максимум 5 шагов);
- сообщение ассистента с `tool_calls` добавляется в `messages` **один раз до цикла** по вызовам (критично!);
- результат инструмента добавляется с ролью `"tool"` и полем `"tool_name"`;
- `temperature = 0.2`;
- `message.get("content") or ""` (защита от `null`);
- обработка ошибок `ConnectionError`, `HTTPError`, `Timeout`.

#### 3.4.2. `critic_agent(question: str, answer: str, model: str) -> str`

**Роль:** Критик. Ищет ошибки, галлюцинации, слишком категоричные советы.

**Требование проекта:** Критик **имеет доступ к `lookup_glossary`** и сверяет определения терминов с глоссарием. Если расходится — пишет:

```
Определение термина «X» расходится с глоссарием:
в ответе — «...», в глоссарии — «...».
```

**Системный промпт:** строгий, но справедливый. Не переписывает ответ — только перечисляет проблемы. Если проблем нет — «Проблем не найдено».

**Логика tool calling:** та же, что в `answer_agent`.

#### 3.4.3. `corrector_agent(question: str, answer: str, critique: str, model: str) -> str`

**Роль:** Корректор. Переписывает ответ с учётом замечаний Критика.

**Требования:**
- не добавляет фактов, которых не было в исходном ответе или глоссарии;
- если Критик сказал «Проблем не найдено» — возвращает исходный ответ без изменений.

#### 3.4.4. `judge_agent(question: str, answer: str, final_answer: str, model: str) -> dict`

**Роль:** Судья. Оценивает, насколько итоговый ответ лучше исходного.

**Возвращает:**

```python
{"score": int, "reason": str}
```

- `score` — от 1 до 10 (1–5 — не лучше, 6–10 — лучше);
- `reason` — обоснование одним предложением.

**Обработка ошибок:** если модель вернула не JSON — `score = 0`, `reason = сырой ответ`.

### 3.5. Файл `multi_agent_core.py` — пайплайн

**Назначение:** импортируемая логика без `print()` и `input()`.

**Обязательная функция:**

```python
def run_pipeline(question: str, model: str) -> dict:
    """
    Возвращает:
    {
        "question": str,
        "answer": str,
        "answer_steps": list,
        "critique": str,
        "final": str,
        "judge": {"score": int, "reason": str},
        "changed": bool
    }
    """
```

**Требования:**
- если Ответчик не справился (`answer.startswith("Ответчик не смог")`) — Критик, Корректор и Судья не вызываются;
- все ошибки пробрасываются наверх (обрабатываются в UI);
- `temperature = 0.2` для всех агентов.

### 3.6. Файл `app.py` — Streamlit-интерфейс

**Назначение:** только UI. Никакой логики — только вызовы из `multi_agent_core.py` и `tools.py`.

#### 3.6.1. Сайдбар

| Элемент | Тип | Поведение |
|---|---|---|
| Заголовок «⚙️ Настройки» | `st.header` | — |
| Выбор модели | `st.selectbox` | Список из `GET /api/tags`, отфильтровать имена с `embed`. Fallback: `["qwen2.5:3b", "qwen2.5:7b"]` |
| «Показывать шаги Ответчика» | `st.checkbox` | По умолчанию `True` |
| «Вызывать судью» | `st.checkbox` | По умолчанию `True` |
| «Очистить историю» | `st.button` | Сбрасывает `session_state.messages` к одному системному промпту |

#### 3.6.2. Основная область

- **Заголовок:** «🧠 Мультиагентный ассистент психолога».
- **Подзаголовок:** тема проекта (например, «Когнитивные искажения»).
- **Дисклеймер (обязательно):** `st.caption("⚠️ Ассистент не заменяет консультацию специалиста. Ответы носят информационный характер.")`.
- **История чата:** `st.chat_message("user")` / `st.chat_message("assistant")`, хранение в `session_state.messages`.
- **Поле ввода:** `st.chat_input("Ваш вопрос:")`.

#### 3.6.3. Отображение ответа — три колонки

```
┌─────────────────┬─────────────────┬─────────────────┐
│  1. Ответчик    │   2. Критик     │  3. Корректор   │
│                 │                 │                 │
│ [ответ]         │ [замечания]     │ [итоговый]      │
│                 │                 │                 │
│ ▼ 🔧 Шаги       │                 │                 │
│ (tool calls)    │                 │                 │
└─────────────────┴─────────────────┴─────────────────┘
```

**Требования:**
- `st.columns(3)`;
- под Ответчиком — `st.expander("🔧 Шаги")` с tool calls (имя, аргументы, результат);
- если `final != answer` — `st.success("Корректор улучшил ответ")`;
- если `final == answer` — `st.info("Критик не нашёл проблем")`;
- если судья включён — `st.metric("Оценка судьи", f"{score}/10")` + `st.caption(reason)`.

#### 3.6.4. Вкладка «📊 Оценка»

- `st.tabs(["💬 Чат", "📊 Оценка"])`.
- Загружает `test_questions.json`.
- Кнопка «Прогнать все».
- Для каждого вопроса — прогон `run_pipeline`.
- `pandas.DataFrame` с колонками: Вопрос | Найден | Критик нашёл | Корректор изменил | Оценка судьи.
- Итог: `st.metric("Средняя оценка", f"{avg}/10")`, `st.metric("Доля случаев, где Критик нашёл проблемы", f"{pct}%")`.

**Важно про HF-демо:** вкладка «Оценка» требует доступа к Ollama, которого на HF Space нет. Поэтому:

- в **локальной версии** — вкладка работает полностью;
- в **демо-версии на HF** — вкладка **скрыта** (если `localhost:11434` недоступен) или показывает сообщение: `«Оценка доступна только при локальном запуске. См. README.»`

Это же поведение должно быть описано в `README.md`.

#### 3.6.5. Обработка ошибок (обязательно)

| Ситуация | Что показываем |
|---|---|
| `ConnectionError` | `st.error("Ollama не запущена. Запустите приложение Ollama.")` |
| HTTP 404 | `st.error` с подсказкой `ollama pull qwen2.5:3b` |
| `glossary.json` не найден | `st.warning("glossary.json не найден в data/")` |
| `notes/` не найдена | `st.warning("Папка data/notes/ не найдена")` |
| Пустой ввод | игнорируем |
| Судья вернул не JSON | `score=0`, `reason=сырой ответ` |

#### 3.6.6. Пустое состояние

Если сообщений нет — приветствие + 4 кнопки-примера вопросов по теме. Нажатие — сразу отправляет вопрос.

#### 3.6.7. Требования к коду

- `@st.cache_data` на загрузку `glossary.json`;
- `st.cache_data.clear()` после редактирования данных (если реализовано);
- комментарии на русском;
- без классов;
- `streamlit run app.py` — команда запуска.

### 3.7. Файл `test_questions.json`

**Формат:**

```json
[
  {
    "question": "Что такое катастрофизация?",
    "expected_keywords": ["катастрофизация", "преувеличение"]
  }
]
```

**Требования:**
- **5–10 вопросов** по теме;
- для каждого — `question` и `expected_keywords` (1–3 ключевых слова);
- **минимум 3 вопроса** на глоссарий (проверка `lookup_glossary`);
- если реализован `add_numbers` — минимум 1 арифметический вопрос; если нет — арифметические вопросы не добавляем (чтобы не было противоречий).

### 3.8. Файл `requirements.txt`

```
streamlit==1.40.0
requests==2.32.0
pandas==2.2.0
```

**Если реализован `parse_source.py`** — добавьте строку:

```
pypdf==5.1.0
```

Если парсер не реализован — `pypdf` **не добавляйте**. Версии — актуальные на момент сдачи. Без лишних библиотек.

### 3.9. Дополнительные требования к коду

- **Комментарии на русском**, короткие, для студента-психолога с базовым Python.
- **Без ООП**, без декораторов (кроме `@st.cache_data`), без асинхронности.
- **Обработка ошибок** — везде, где возможны сбои (сеть, файлы, JSON).
- **Никаких `print()`** в `multi_agent_core.py` и `tools.py` (только в `test_tools.py` и `parse_source.py`).
- **Все пути** — относительные (`data/glossary.json`, а не `C:/...`).

---

## 4. Требования к Git

### 4.1. Репозиторий

- Публичный репозиторий на GitHub.
- Название: `psych-assistant-<тема>` (латиницей, без пробелов).

### 4.2. Коммиты

**Требования для индивидуального проекта:**
- **8–10 осмысленных коммитов**;
- сообщения **описывают, что сделано**, а не «update»/«fix»;
- коммиты **равномерно по времени** (не всё в последнюю ночь).

**Требования для парного проекта (обязательно):**
- **6–8 осмысленных коммитов от каждого участника** (итого 12–16 на репозиторий);
- **каждый участник** должен иметь коммиты минимум в **трёх разных смысловых зонах**: `data/` или `tools.py`, `agents.py` или `multi_agent_core.py`, `app.py` или `README.md`. Это доказывает, что оба работали над проектом, а не один писал код, а второй «числился»;
- коммиты **равномерно по времени** у каждого участника;
- проверка: `git log --oneline --author="Имя"` — у каждого должно быть 6–8 строк.

**Пример правильной последовательности (для пары):**

```bash
# Участник А — данные и инструменты
git add data/ glossary.json
git commit -m "Шаг 1: собран корпус по теме когнитивные искажения, 12 терминов"

git add parse_source.py data/source.txt
git commit -m "Шаг 1: парсер PDF→txt, извлечено 12000 символов"

git add tools.py
git commit -m "Шаг 2: реализован lookup_glossary"

git add tools.py test_tools.py
git commit -m "Шаг 2: добавлен read_notes и 5 тестов"

git add agents.py
git commit -m "Шаг 3: критик сверяется с глоссарием через lookup_glossary"

git add app.py
git commit -m "Шаг 4: добавлены кнопки-примеры в пустое состояние"

# Участник Б — агенты и интерфейс
git add agents.py
git commit -m "Шаг 3: реализован answer_agent с tool calling"

git add agents.py multi_agent_core.py
git commit -m "Шаг 3: добавлены corrector_agent, judge_agent и run_pipeline"

git add app.py requirements.txt
git commit -m "Шаг 4: Streamlit с тремя колонками, дисклеймером и вкладкой оценки"

git add app.py
git commit -m "Шаг 4: обработка ConnectionError и HTTP 404 в UI"

git add README.md REPORT.md
git commit -m "Шаг 7: README с вкладом участников и отчёт с рефлексией"

# Совместный коммит (по договорённости — кто-то один пушит)
git add .
git commit -m "Шаг 7: финальная проверка перед деплоем на HF"
```

**Плохие сообщения (не принимаются):** `update`, `fix`, `asdf`, `.`.

### 4.3. Проверка

```bash
git log --oneline --author="Имя участника"
```

У каждого участника пары — 6–8 строк с осмысленными сообщениями, минимум в трёх разных зонах (`data`/`tools`, `agents`/`multi_agent_core`, `app`/`README`).

**Если проверка не проходит** — это фиксируется в критерии «Git» (15%) и в критерии «Защита» (10%).

---

## 5. Требования к публикации на Hugging Face Spaces

### 5.1. Вариант А (рекомендуемый): демо-версия с заглушкой

**Проблема:** HF Space — облачный сервер, он не имеет доступа к `http://localhost:11434` на компьютере студента.

**Решение:**
- в `app.py` — проверка: если `localhost:11434` недоступен, показать демо-ответ («Это демо-версия. Полноценный ассистент работает локально, см. README»);
- вкладка «📊 Оценка» в демо-версии **скрыта** или показывает сообщение «доступно локально»;
- в `README.md` — объяснение: «Локальная Ollama недоступна из облака, поэтому на HF развёрнута демо-версия интерфейса. Полная версия с работающими агентами и оценкой — локально».

### 5.2. Вариант Б (продвинутый): ngrok

- локальная Ollama пробрасывается в интернет через ngrok;
- URL хранится в секретах HF Space;
- в `app.py` — обращение к этому URL вместо `localhost`.

### 5.3. Пошаговая публикация

1. Аккаунт на [huggingface.co](https://huggingface.co).
2. Создать Space: [huggingface.co/new-space](https://huggingface.co/new-space) → SDK **Streamlit** → Public.
3. Склонировать: `git clone https://huggingface.co/spaces/<username>/<space-name>`.
4. Скопировать `app.py`, `multi_agent_core.py`, `tools.py`, `agents.py`, `data/`, `requirements.txt`.
5. Отправить:
```bash
git add .
git commit -m "Deploy: демо-версия для Hugging Face Spaces"
git push
```
6. Дождаться сборки (2–5 минут), открыть ссылку.

### 5.4. Что должно быть на HF

- `app.py`
- `multi_agent_core.py`
- `agents.py`
- `tools.py`
- `data/` (глоссарий и заметки)
- `requirements.txt`
- `README.md` (с объяснением про демо-версию и скрытую вкладку оценки)

---

## 6. Требования к документации

### 6.1. `README.md` (полстраницы)

**Обязательные разделы:**

1. **Название проекта и тема.**
2. **Источник данных** — название книги / ссылки на статьи / описание опроса, с указанием лицензии или права доступа.
3. **Вклад участников** (если пара) — **кто за какие этапы отвечал** (данные / парсер / инструменты / агенты / интерфейс / деплой / документация). Формулировка «делали всё вместе» не принимается — нужны конкретные этапы.
4. **Команды запуска:**
   ```
   pip install -r requirements.txt
   streamlit run app.py
   ```
5. **Объяснение, почему на HF демо-версия** (если Вариант А) и почему вкладка «Оценка» на HF не работает.
6. **3 примера работы:** вопрос → ответ трёх агентов → скриншот.
7. **Структура проекта** (дерево файлов).
8. **Дисклеймер:** ассистент не заменяет консультацию специалиста.

**Шаблон структуры (ориентир, не обязателен дословно):**

```markdown
# Мультиагентный ассистент психолога: <тема>

## Источник данных
...

## Вклад участников
- Участник А: данные, глоссарий, инструменты, тесты
- Участник Б: агенты, пайплайн, Streamlit, деплой

## Запуск
...

## Демо-версия на HF
...

## Примеры работы
...

## Структура проекта
...

## Дисклеймер
...
```

### 6.2. `REPORT.md` (1 страница)

**Обязательные разделы:**

1. **Что получилось** — краткое описание работающего проекта.
2. **В каких случаях Критик находил проблемы, в каких — нет** — 2–3 конкретных примера.
3. **Где цепочка работала хорошо, где — плохо** — честная рефлексия.
4. **Что бы изменили в системных промптах** — 1–2 конкретных предложения.
5. **Что бы изменили в данных** (глоссарий, заметки, источник) — 1 конкретное предложение. Например: «добавил бы больше терминов про когнитивные ошибки в социальных ситуациях, потому что Критик часто не мог свериться с глоссарием».

Пункт 5 обязателен — он показывает, что студент думает не только про код, но и про содержание.

---

## 7. Что сдавать (чек-лист)

```
<репозиторий>/
├── data/
│   ├── source.pdf (или .txt)      ← первоисточник (легальный)
│   ├── glossary.json               ← 10+ терминов
│   ├── notes/                      ← 3–5 файлов
│   │   ├── definition.txt
│   │   ├── examples.txt
│   │   └── ...
│   └── chunks.jsonl                ← опционально
├── parse_source.py                 ← опционально (парсер PDF→txt/jsonl)
├── tools.py                        ← инструменты
├── test_tools.py                   ← 5+ тестов (6, если есть add_numbers)
├── agents.py                       ← 4 агента
├── multi_agent_core.py             ← пайплайн
├── app.py                          ← Streamlit + дисклеймер
├── test_questions.json             ← 5–10 вопросов
├── requirements.txt
├── README.md                       ← полстраницы, вклад по этапам
└── REPORT.md                       ← 1 страница, включая рефлексию по данным
```

**Ссылки для сдачи:**
1. URL репозитория на GitHub.
2. URL Space на Hugging Face (или скриншоты + объяснение в README).

---

## 8. Критерии приёмки

| Критерий | Вес | Что проверяется |
|---|---|---|
| **Сбор и качество данных** | 15% | Релевантность теме, легальность источника, 10+ терминов, JSON валиден, источник уникален |
| **Работа мультиагентной системы** | 25% | Все 4 роли работают (включая Судью), Ответчик вызывает инструменты, Критик **сверяется с глоссарием** |
| **Streamlit-интерфейс** | 15% | Запускается, три колонки, tool calls видны, дисклеймер, ошибки обработаны, вкладка «Оценка» корректно скрыта на HF |
| **Git** | 15% | Индивидуально: 8–10 коммитов. Пара: 6–8 коммитов **от каждого** в **3+ зонах** (`data`/`tools`, `agents`/`core`, `app`/`README`), равномерно по времени |
| **Публикация на HF** | 10% | Space доступен (или объяснение в README, почему демо), вкладка «Оценка» скрыта корректно |
| **README и отчёт** | 10% | Полнота, рефлексия (включая пункт про данные), вклад участников **по этапам**, дисклеймер |
| **Защита** | 10% | Объясняет свой код, код партнёра (если пара), легальность источника |

**Минимальный порог:** 60% — проект засчитан. Меньше — доработка.

---

## 9. Сроки

| Неделя | Что делаем | Что сдаём |
|---|---|---|
| 1 | Реестр тем, выбор источника | Запись в реестре с темой и источником (с лицензией) |
| 2 | Сбор корпуса, парсер (опц.), глоссарий | `data/` |
| 3 | `tools.py`, `test_tools.py`, начало `agents.py` | Коммиты в `tools.py`, `agents.py` |
| 4 | Завершение агентов, Критик с глоссарием, пайплайн | `agents.py`, `multi_agent_core.py` |
| 5 | Streamlit + дисклеймер + вкладка оценки | `app.py` |
| 6 | Отладка, локальная версия работает | Работающий проект |
| 7 | HF Space (демо), README, отчёт | Ссылка + `README.md` + `REPORT.md` |
| 8 | Защита | Отчёт, защита |

**Жёсткие дедлайны:**
- конец 1-й недели — тема занята (без темы проект не принимается);
- конец 6-й — рабочая локальная версия;
- конец 7-й — ссылка на HF;
- 8-я неделя — защита.

---

## 10. Защита

**8–10 минут на студента, 12–15 минут на пару.**

Преподаватель:

1. **Открывает любой файл** из репозитория и просит объяснить конкретную строку. Вопросы **по существу**, а не на память:
   - «Почему `ConnectionError` обрабатывается отдельно от `HTTPError`?» (обработка ошибок);
   - «Почему `steps` — список словарей, а не плоский текст?» (структура данных);
   - «Почему `temperature = 0.2`, а не 0.7?» (выбор параметров);
   - «Почему сообщение ассистента с `tool_calls` добавляется в историю до цикла по вызовам?» (логика агента).

   Если студент не помнит точную строчку, но понимает **принцип** — это засчитывается. Если не понимает принцип — нет.

2. **Задаёт вопрос не из README** — проверка, что система работает, а не захардкожена.

3. **Спрашивает про легальность источника:** «Откуда взяли `source.pdf`? Какая лицензия / право доступа?»

4. **Просит показать `git log --oneline --author="..."`** — для пары у каждого участника должно быть 6–8 коммитов в 3+ зонах.

5. **Для пары:** просит объяснить, что делал партнёр (по этапам из README). Если студент не знает, чем занимался партнёр — это фиксируется.

6. **Просит продемонстрировать всех четырёх агентов** (включая Судью — чекбокс «Вызывать судью» включён). Если чекбокс был выключен на защите — студент должен включить и показать.

**Что важно:** студент отвечает и за свои файлы, и за общий `app.py`. Если не может объяснить свой код — фиксируется.

---

## 11. Что получит студент

По итогам проекта:

1. **Работающий мультиагентный ассистент** по выбранной психологической теме.
2. **Публичная ссылка** на Hugging Face Space (демо-версия).
3. **Репозиторий на GitHub** с историей разработки (коммиты от каждого участника).
4. **Навыки:**
   - сбор и структурирование научных данных;
   - парсинг PDF;
   - промпт-инжиниринг (системные промпты для четырёх ролей);
   - tool calling и мультиагентные системы;
   - Streamlit-интерфейс;
   - Git (контроль версий, работа в паре);
   - деплой в облако (HF Spaces).
5. **Проект в портфолио** — не учебная поделка, а реальный продукт.

---

## 12. Академическая честность

**Разрешено:**
- использовать ИИ-ассистентов для генерации кода и объяснений;
- консультироваться с преподавателем и другими студентами;
- использовать открытые источники и документацию;
- скачивать open-access и public-domain материалы, писать парсеры.

**Запрещено:**
- копировать чужой репозиторий (даже частично);
- выдавать чужой код за свой;
- «занимать» тему и не работать над ней;
- выкладывать в публичный репозиторий пиратские PDF и закрытые статьи;
- в паре: один участник пишет весь код, второй только «числится» (проверяется по `git log --author` и защите).


